# BÁO CÁO PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)
## Đề tài: Khảo sát bài toán & Phân tích bộ dữ liệu Ames Housing
---
### Mục tiêu báo cáo:
1. **[1] Đọc paper & Khảo sát bài toán:**
2. **[2] Phân tích khám phá dữ liệu (EDA):**
   + Xác định tính chất
   + Phân tích đơn biến
   + Phân tích đa biến
   + Xác định giá trị thiếu, và mẫu bất thường
   + Xác định ngoại lệ

---
## PHẦN 1: Xác định tính chất
### 1.1. Bối cảnh từ Paper Dean De Cock (2011)
* **Lý do thay thế Boston Housing:** Bộ dữ liệu Boston Housing (1978) đã quá cũ kỹ (từ những năm 1970), quy mô nhỏ (chỉ 506 mẫu và 14 biến), giá nhà không còn thực tế đối với sinh viên hiện đại.
* **Nguồn gốc dữ liệu Ames:** Dữ liệu trích xuất từ **Văn phòng Thẩm định Giá Thành phố Ames (Ames City Assessor's Office)**, Iowa, ghi nhận các giao dịch nhà ở từ năm 2006 đến 2010.
* **Bài toán Kaggle:**
  * Thuộc lớp bài toán **Học có giám sát**, dạng **Hồi quy**.
  * **Biến mục tiêu (Target $Y$):** `SalePrice` (Giá bán của ngôi nhà bằng USD).
  * **Thước đo đánh giá (Evaluation Metric):** Đánh giá dựa trên sai số căn bậc hai trung bình logarit (**RMSLE**):
    $$\text{RMSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^n \left( \log(\hat{y}_i) - \log(y_i) \right)^2}$$
    *Ý nghĩa:* Giúp đánh giá sai số theo tỷ lệ phần trăm chênh lệch, tạo sự công bằng giữa việc dự đoán các ngôi nhà giá rẻ và nhà giá cao.

In [34]:
# Import các thư viện cần thiết
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

# Đọc tập dữ liệu
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Kích thước tập Train: {train.shape}")
print(f"Kích thước tập Test:  {test.shape}")

Kích thước tập Train: (1460, 81)
Kích thước tập Test:  (1459, 80)


In [35]:
# Xem qua 5 dòng đầu tiên của tập dữ liệu
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


### 1.2. Phân loại 79 đặc trưng theo Paper Dean De Cock
Theo tài liệu kỹ thuật của tác giả, 79 biến dự đoán được chia thành 4 nhóm cụ thể:
1. **20 Biến liên tục (Continuous):** Đo đạc kích thước diện tích (`LotFrontage`, `LotArea`, `TotalBsmtSF`, `GrLivArea`, `GarageArea`, `WoodDeckSF`,...).
2. **14 Biến rời rạc (Discrete):** Đếm số lượng phòng và năm xây dựng/cải tạo (`YearBuilt`, `YearRemodAdd`, `FullBath`, `HalfBath`, `BedroomAbvGr`, `TotRmsAbvGrd`, `GarageCars`,...).
3. **23 Biến định danh (Nominal):** Phân loại không có thứ bậc (`Neighborhood`, `MSZoning`, `BldgType`, `HouseStyle`, `Foundation`, `SaleType`, `SaleCondition`,...).
4. **23 Biến thứ bậc (Ordinal):** Đánh giá theo thang bậc chất lượng (`OverallQual`, `OverallCond`, `ExterQual`, `BsmtQual`, `KitchenQual`, `FireplaceQu`, `GarageQual`,...).

In [36]:
# Thống kê tổng số lượng cột số (Numerical) và cột định tính (Categorical)
num_features = train.select_dtypes(include=[np.number]).columns.drop(['Id', 'SalePrice'])
cat_features = train.select_dtypes(include=['object']).columns

print(f"Số lượng biến số (Numerical): {len(num_features)}")
print(f"Số lượng biến định tính (Categorical): {len(cat_features)}")

Số lượng biến số (Numerical): 36
Số lượng biến định tính (Categorical): 43


---
## PHẦN 2: PHÂN TÍCH ĐƠN BIẾN
### 2.1. Phân tích biến mục tiêu `SalePrice`
Ta kiểm tra các chỉ số thống kê mô tả cơ bản của `SalePrice` và kiểm tra độ lệch (Skewness) cùng độ nhọn (Kurtosis).

In [37]:
# Thống kê mô tả biến mục tiêu SalePrice
print("Thống kê mô tả SalePrice:")
print(train['SalePrice'].describe().apply(lambda x: f"{x:,.2f}"))
print(f"Độ lệch (Skewness): {train['SalePrice'].skew():.2f}")
print(f"Độ nhọn (Kurtosis): {train['SalePrice'].kurt():.2f}")

Thống kê mô tả SalePrice:
count      1,460.00
mean     180,921.20
std       79,442.50
min       34,900.00
25%      129,975.00
50%      163,000.00
75%      214,000.00
max      755,000.00
Name: SalePrice, dtype: str
Độ lệch (Skewness): 1.88
Độ nhọn (Kurtosis): 6.54
